# Passenger Analysis

In [2]:
import duckdb

In [3]:
df = duckdb.read_parquet('../raw/*/*.parquet')
df

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ airport_fee │
│  int64   │      timestamp       │       timestamp       │     double      │    double     │   double   │      varchar       │    int64     │    int64     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

## Checking what all Uniquevalues are there in Passenger_count row

In [4]:
duckdb.sql("""
            SELECT 
                DISTINCT passenger_count
            FROM 
                df
            """)

┌─────────────────┐
│ passenger_count │
│     double      │
├─────────────────┤
│             2.0 │
│             3.0 │
│             9.0 │
│             4.0 │
│             0.0 │
│             5.0 │
│             8.0 │
│             7.0 │
│            NULL │
│             1.0 │
│             6.0 │
└─────────────────┘
      11 rows    

## Total number of rows where passenger count is NULL

In [5]:
duckdb.sql("""
            SELECT 
                COUNT (*) AS null_passesnger_count
            FROM
               df
            WHERE
                passenger_count IS NULL

            """)

┌───────────────────────┐
│ null_passesnger_count │
│         int64         │
├───────────────────────┤
│              18380785 │
└───────────────────────┘

**There are more than `18.3` million rows where passenger count is `NULL`**

In [6]:
nulp_df = duckdb.sql("""
                    SELECT 
                        tpep_pickup_datetime,
                        tpep_dropoff_datetime,
                        passenger_count,
                        payment_type,
                        total_amount
                    FROM
                        df
                    WHERE
                        passenger_count IS NULL
                    """)
nulp_df

┌──────────────────────┬───────────────────────┬─────────────────┬──────────────┬──────────────┐
│ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ payment_type │ total_amount │
│      timestamp       │       timestamp       │     double      │    int64     │    double    │
├──────────────────────┼───────────────────────┼─────────────────┼──────────────┼──────────────┤
│ 2022-01-01 00:50:00  │ 2022-01-01 00:54:00   │            NULL │            0 │        18.25 │
│ 2022-01-01 00:49:24  │ 2022-01-01 01:27:36   │            NULL │            0 │        55.72 │
│ 2022-01-01 00:42:00  │ 2022-01-01 00:56:00   │            NULL │            0 │        20.04 │
│ 2022-01-01 00:40:00  │ 2022-01-01 00:55:00   │            NULL │            0 │        21.21 │
│ 2022-01-01 00:40:00  │ 2022-01-01 00:52:00   │            NULL │            0 │        21.75 │
│ 2022-01-01 00:18:10  │ 2022-01-01 00:30:29   │            NULL │            0 │         15.3 │
│ 2022-01-01 00:33:22  │ 2022-

## Checking the unique values in `payment_type` where passenger count is NULL

In [7]:
duckdb.sql("""
            SELECT 
                DISTINCT payment_type
            FROM 
                nulp_df
            """)

┌──────────────┐
│ payment_type │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

**All the payment types is `0` when passenger_count is NULL**

Payment type '0' = `Flex Fare trip`  

`one reason i came up with was that since it is usually booked through the application they couldn't record the number of passengers`

What is Flex Fare Trip?

`Flex Fare Trip is an E-Hail requested through a TLC-licensed E-Hail application for yellow
taxicabs and Street Hail Liveries with an upfront, non-metered, binding fare in dollars and
cents, inclusive of all taxes, fees, and any price multiplier, for the specific trip requested, prior to
dispatching the trip.`

`NOTE`: E-Hail is a Hail requested through an E-Hail Application, which can either be a metered fare
trip or a Flex Fare trip. 

In [9]:
duckdb.sql("""
            SELECT 
               COUNT (total_amount)
            FROM 
                nulp_df
            WHERE
                total_amount < 0
            """)

┌─────────────────────┐
│ count(total_amount) │
│        int64        │
├─────────────────────┤
│              170659 │
└─────────────────────┘

In [10]:
duckdb.sql("""
            SELECT
                passenger_count,
                payment_type
            FROM
                df
            """)

┌─────────────────┬──────────────┐
│ passenger_count │ payment_type │
│     double      │    int64     │
├─────────────────┼──────────────┤
│             2.0 │            1 │
│             1.0 │            1 │
│             1.0 │            1 │
│             1.0 │            2 │
│             1.0 │            1 │
│             1.0 │            1 │
│             1.0 │            1 │
│             1.0 │            2 │
│             1.0 │            1 │
│             1.0 │            2 │
│              ·  │            · │
│              ·  │            · │
│              ·  │            · │
│             5.0 │            2 │
│             5.0 │            2 │
│             5.0 │            2 │
│             5.0 │            1 │
│             2.0 │            1 │
│             2.0 │            1 │
│             4.0 │            1 │
│             1.0 │            1 │
│             2.0 │            1 │
│             1.0 │            1 │
└─────────────────┴──────────────┘
  ? rows            

In [21]:
duckdb.sql("""
            SELECT 
                passenger_count,
                payment_type,
                COUNT(*) AS ride_count
            FROM
                df
            GROUP BY
                    passenger_count, 
                    payment_type
            ORDER BY
                payment_type
            """).df()

,passenger_count,payment_type,ride_count
0,NaN,0,18380785
1,9.0,1,141
2,7.0,1,349
3,3.0,1,4200610
4,5.0,1,1348702
5,6.0,1,881371
6,0.0,1,1576297
7,1.0,1,93709382
8,4.0,1,2260573
9,8.0,1,575


In [25]:
test1 = duckdb.sql("""
                SELECT
                    passenger_count,
                    payment_type
                FROM
                    df
                WHERE
                    payment_type = 0
                """)

In [27]:
duckdb.sql("""
            SELECT
                DISTINCT
                    passenger_count
            FROM
                test1
            """)

┌─────────────────┐
│ passenger_count │
│     double      │
├─────────────────┤
│            NULL │
└─────────────────┘

**From the above analysis payment type `0` is only used when the passenger count is `NULL`**

## References

https://www.nyc.gov/assets/tlc/downloads/pdf/flex_fare_rule_package_08_14_24.pdf